# 01 — Demand Patterns
**Question:** When is demand highest, and how do weekday vs weekend patterns differ?

Data source: `mart_station_demand` and `fact_rentals` in `m2-dataset-study.london_bicycles`

In [3]:
from google.cloud import bigquery
from sqlalchemy import create_engine
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

PROJECT = 'm2-dataset-study'
DATASET = 'london_bicycles'

engine = create_engine(f'bigquery://{PROJECT}/{DATASET}')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

ModuleNotFoundError: No module named 'sqlalchemy'

## 1.1 — Hourly demand heatmap (weekday vs weekend)

In [2]:
sql = """
SELECT
    hour_of_day,
    day_type,
    SUM(departures) AS total_departures
FROM `m2-dataset-study.london_bicycles.mart_station_demand`
GROUP BY hour_of_day, day_type
"""
df = pd.read_sql(sql, engine)
pivot = df.pivot(index='hour_of_day', columns='day_type', values='total_departures')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, cmap in zip(axes, ['Weekday', 'Weekend'], ['Blues', 'Oranges']):
    vals = pivot[[col]]
    sns.heatmap(vals, ax=ax, cmap=cmap, fmt=',', annot=True,
                linewidths=0.5, cbar_kws={'label': 'Departures'})
    ax.set_title(f'{col} Hourly Departures (full dataset)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Hour of Day')
    ax.set_xlabel('')

plt.suptitle('Two Markets, One Fleet — Bimodal Weekday vs Flat Weekend', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/01_demand_heatmap.png', bbox_inches='tight')
plt.show()

NameError: name 'pd' is not defined

## 1.2 — Peak hour comparison chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
hours = range(24)
weekday = pivot.get('Weekday', pd.Series(0, index=range(24)))
weekend = pivot.get('Weekend', pd.Series(0, index=range(24)))

ax.fill_between(hours, weekday, alpha=0.6, color='steelblue', label='Weekday')
ax.fill_between(hours, weekend, alpha=0.6, color='darkorange', label='Weekend')
ax.plot(hours, weekday, color='steelblue', linewidth=2)
ax.plot(hours, weekend, color='darkorange', linewidth=2)

ax.axvspan(7.5, 9.5, alpha=0.1, color='red', label='AM commute peak')
ax.axvspan(16.5, 19.0, alpha=0.1, color='purple', label='PM commute peak')

ax.set_xticks(range(24))
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Total Departures (all years)', fontsize=12)
ax.set_title('Weekday vs Weekend Demand — Two Distinct Shapes', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../outputs/01_peak_comparison.png', bbox_inches='tight')
plt.show()

## 1.3 — Trip segment breakdown

In [ ]:
seg_sql = """
SELECT
    trip_segment,
    COUNT(*) AS trip_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM `m2-dataset-study.london_bicycles.fact_rentals`
GROUP BY trip_segment
ORDER BY trip_count DESC
"""
seg = pd.read_sql(seg_sql, engine)

colours = {'commute_am': '#2E86AB', 'commute_pm': '#A23B72',
           'leisure': '#F18F01', 'nightlife': '#6C3483',
           'night': '#4A4E69', 'daytime': '#95A5A6'}

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(seg['trip_segment'], seg['trip_count'],
               color=[colours.get(s, '#888') for s in seg['trip_segment']])
for bar, pct in zip(bars, seg['pct']):
    ax.text(bar.get_width() + 2e5, bar.get_y() + bar.get_height()/2,
            f'{pct}%', va='center', fontsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
ax.set_xlabel('Total Trips (full dataset)', fontsize=12)
ax.set_title('Trip Segment Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/01_segment_breakdown.png', bbox_inches='tight')
plt.show()
print(seg.to_string(index=False))